<a href="https://colab.research.google.com/github/stephanie465337/Data-Science-Portfolio-C21/blob/main/NotebookLM/Module%201/8e_Pandas_Nested_to_Wide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nested to Wide and Back

There are three ways to get wide data from a nested structure.
1. boolean ( True/False ) with the categories as columns
1. counts with the categories as columns
1. categories with counts as columns


From https://stackoverflow.com/a/69604744

In [ ]:
import pandas as pd
import numpy as np

Create a data frame with two columns, the seconds of which has nested lists, sometimes with duplicated entries.

Notice that "apple" appears twice in the second column of the first row.

In [ ]:
data = pd.DataFrame()
data['id'] = ["ab3e3", "psdds2", "pas13", "ccdf2", "dsda1"]
data['fruit'] = ["apple, orange, apple", "others", "dragon fruit, orange", "watermelon", "others"]

df = pd.DataFrame(data)
df

,id,fruit
0,ab3e3,"apple, orange, apple"
1,psdds2,others
2,pas13,"dragon fruit, orange"
3,ccdf2,watermelon
4,dsda1,others


In [ ]:
print(df.to_markdown(index=False))

| id     | fruit                |
|:-------|:---------------------|
| ab3e3  | apple, orange, apple |
| psdds2 | others               |
| pas13  | dragon fruit, orange |
| ccdf2  | watermelon           |
| dsda1  | others               |


## Nested to Wide

### Boolean: Using .get_dummies() with a delimeter

Expand column categories into individual columns with binary values.

In [ ]:
df['fruit'].str.get_dummies(', ')

,apple,dragon fruit,orange,others,watermelon
0,1,0,1,0,0
1,0,0,0,1,0
2,0,1,1,0,0
3,0,0,0,0,1
4,0,0,0,1,0


Join original data frame with get_dummies.

In [ ]:
df.join(df['fruit'].str.get_dummies(', '))

,id,fruit,apple,dragon fruit,orange,others,watermelon
0,ab3e3,"apple, orange, apple",1,0,1,0,0
1,psdds2,others,0,0,0,1,0
2,pas13,"dragon fruit, orange",0,1,1,0,0
3,ccdf2,watermelon,0,0,0,0,1
4,dsda1,others,0,0,0,1,0


Join then drop "fruit".

In [ ]:
df_wide1 = (
  df
  .join(df['fruit'].str.get_dummies(', '))
  .drop(columns= ['fruit'])
)
df_wide1

,id,apple,dragon fruit,orange,others,watermelon
0,ab3e3,1,0,1,0,0
1,psdds2,0,0,0,1,0
2,pas13,0,1,1,0,0
3,ccdf2,0,0,0,0,1
4,dsda1,0,0,0,1,0


Drop "fruit" then join.


In [ ]:
(
df
  .drop(columns=['fruit'])
  .join(df['fruit'].str.get_dummies(', '))
)

,id,apple,dragon fruit,orange,others,watermelon
0,ab3e3,1,0,1,0,0
1,psdds2,0,0,0,1,0
2,pas13,0,1,1,0,0
3,ccdf2,0,0,0,0,1
4,dsda1,0,0,0,1,0


# Using .explode()

Explode a data series.

In [ ]:
df_explode = df.assign(fruit=df["fruit"].str.split(", ")).explode("fruit")
df_explode

,id,fruit
0,ab3e3,apple
0,ab3e3,orange
0,ab3e3,apple
1,psdds2,others
2,pas13,dragon fruit
2,pas13,orange
3,ccdf2,watermelon
4,dsda1,others


In [ ]:
ct = df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").drop(columns=["id"])
ct

,fruit
0,apple
0,orange
0,apple
1,others
2,dragon fruit
2,orange
3,watermelon
4,others


In [ ]:
ct["fruit"].unique()

array(['apple', 'orange', 'others', 'dragon fruit', 'watermelon'],
      dtype=object)

In [ ]:
ct.value_counts()

,count
fruit,
apple,2
orange,2
others,2
dragon fruit,1
watermelon,1


In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts()

id      fruit       
ab3e3   apple           2
        orange          1
ccdf2   watermelon      1
dsda1   others          1
pas13   dragon fruit    1
        orange          1
psdds2  others          1
Name: count, dtype: int64

In [ ]:
df_counts = df_explode.value_counts().reset_index()
df_counts

,id,fruit,count
0,ab3e3,apple,2
1,ab3e3,orange,1
2,ccdf2,watermelon,1
3,dsda1,others,1
4,pas13,dragon fruit,1
5,pas13,orange,1
6,psdds2,others,1


In [ ]:
np.repeat(df_counts.index, df_counts['count'])

Index([0, 0, 1, 2, 3, 4, 5, 6], dtype='int64')

In [ ]:
df_original = df_counts.reindex(df_counts.index.repeat(df_counts['count']))

In [ ]:
df_counts['count'], df_counts.index.repeat(df_counts['count'])

(0    2
 1    1
 2    1
 3    1
 4    1
 5    1
 6    1
 Name: count, dtype: int64,
 Index([0, 0, 1, 2, 3, 4, 5, 6], dtype='int64'))

In [ ]:
df_counts.iloc[df_counts.index.repeat(df_counts['count'])]

,id,fruit,count
0,ab3e3,apple,2
0,ab3e3,apple,2
1,ab3e3,orange,1
2,ccdf2,watermelon,1
3,dsda1,others,1
4,pas13,dragon fruit,1
5,pas13,orange,1
6,psdds2,others,1


In [ ]:
(
df_counts
  .reindex(df_counts.index.repeat(df_counts['count']))
  .drop(columns='count')
  .reset_index(drop=True)
)

,id,fruit
0,ab3e3,apple
1,ab3e3,apple
2,ab3e3,orange
3,ccdf2,watermelon
4,dsda1,others
5,pas13,dragon fruit
6,pas13,orange
7,psdds2,others


In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts().reset_index()

,id,fruit,count
0,ab3e3,apple,2
1,ab3e3,orange,1
2,ccdf2,watermelon,1
3,dsda1,others,1
4,pas13,dragon fruit,1
5,pas13,orange,1
6,psdds2,others,1


# Using .crosstab()

In [ ]:
pd.crosstab(ct.index, ct["fruit"])

fruit,apple,dragon fruit,orange,others,watermelon
row_0,,,,,
0,2,0,1,0,0
1,0,0,0,1,0
2,0,1,1,0,0
3,0,0,0,0,1
4,0,0,0,1,0


In [ ]:
pd.crosstab(ct.index, ct["fruit"]).rename_axis(None, axis=1)

,apple,dragon fruit,orange,others,watermelon
row_0,,,,,
0,2,0,1,0,0
1,0,0,0,1,0
2,0,1,1,0,0
3,0,0,0,0,1
4,0,0,0,1,0


In [ ]:
pd.crosstab(ct.index, ct["fruit"]).rename_axis(None, axis=1).rename_axis(None, axis=0)

,apple,dragon fruit,orange,others,watermelon
0,2,0,1,0,0
1,0,0,0,1,0
2,0,1,1,0,0
3,0,0,0,0,1
4,0,0,0,1,0


In [ ]:
df.join(pd.crosstab(ct.index, ct["fruit"]).rename_axis(None, axis=1).rename_axis(None, axis=0)).drop( columns = ["fruit"])

,id,apple,dragon fruit,orange,others,watermelon
0,ab3e3,2,0,1,0,0
1,psdds2,0,0,0,1,0
2,pas13,0,1,1,0,0
3,ccdf2,0,0,0,0,1
4,dsda1,0,0,0,1,0
